# Bắt đầu: từ trận đấu đến báo cáo

Dữ liệu thật Ngoại hạng Anh 2025–2026. SQL chạy qua Python.

**Câu hỏi:** Bảng điểm và biểu đồ được tính từ đâu?

**Dự đoán:** trước khi chạy, viết mỗi dòng kết quả đại diện cho gì.

In [ ]:
from pathlib import Path
import os, sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'epl_lab').is_dir())
sys.path.insert(0, str(root))
from epl_lab.learning import database, read_sql
db = database()
print('SQLite chỉ đọc:', db)


### 1. Đọc bảng nguồn
Mỗi dòng matches là một trận, không phải một đội hoặc cầu thủ.

In [ ]:
read_sql('SELECT * FROM matches ORDER BY played_on LIMIT 5')

### 2. Kiểm tra trước khi cộng
Toàn mùa phải có380trận20đội; không tự bỏ dòng sai.

In [ ]:
from epl_lab.analysis import quality, queries, reconcile, scope
quality(db)

### 3. JOIN đội hai lần
Alias h/a cùng đến teams; số dòng vẫn là số trận.

In [ ]:
read_sql('SELECT m.played_on,h.name home,a.name away,m.home_goals,m.away_goals FROM matches m JOIN teams h ON h.team_id=m.home_team_id JOIN teams a ON a.team_id=m.away_team_id LIMIT 10')

### 4. SQL: hai góc nhìn của mỗi trận
UNION ALL giữ cả sân nhà và sân khách. COUNT đội–trận gấp đôi trận nguồn.

In [ ]:
from epl_lab.analysis import team_cte
as_of=scope(db)
read_sql(team_cte()+'SELECT team_id,COUNT(*) played,SUM(points) points FROM team_matches GROUP BY team_id ORDER BY points DESC', {'as_of':as_of})

### 5. pandas tính lại độc lập
Hàm pandas đọc bảng matches/teams rồi tự mở rộng hai phía; không đọc kết quả SQL tổng hợp.

In [ ]:
tables=reconcile(db, as_of)
assert len(tables['team_matches'])==760
tables['standings']

## Bổ sung: kiểm soát bóng trung bình
Chọn menu 7 hoặc chạy chuẩn bị với `--possession` ngoài notebook trước. Trung bình chỉ dùng trận hợp lệ đến mốc; NULL không phải 0. Đối chiếu 379 trận hợp lệ và một xung đột tỷ số; không dùng trung bình toàn mùa cho mốc nửa mùa.


In [ ]:
tables['standings'][['name','avg_possession_pct','possession_matches','possession_missing']].round({'avg_possession_pct':2})

### 6. Phong độ và mốc lịch sử
Chỉ lấy trận đến hết ngày; tối đa5trận/đội, luôn hiển thị số trận.

In [ ]:
queries(db,'2026-01-01')['last5_form']

### 7. Vẽ bốn hình và xuất bảng
Đã đối chiếu xong mới xuất; không thay đổi database.

In [ ]:
from epl_lab.analyze import analyze
out=analyze(db,as_of,root/'outputs/notebook')
from IPython.display import display, Image
for path in sorted(out.glob('*.png')): display(Image(filename=str(path)))

### 8. Đọc ba nhận xét và giới hạn
Điểm từ tỷ số, không phải mọi quyết định hành chính; tương quan không phải nguyên nhân.

In [ ]:
print((out/'report.md').read_text(encoding='utf-8'))

## Tự làm và kiểm tra

Chọn một đội bạn thích, tự kiểm tra GF/GA/điểm từ các trận. Viết3nhận xét gồm số liệu, mốc và giới hạn. Thử thay mốc rồi quan sát thay đổi.

Chỉ mở `docs/solutions.md` sau khi thử. Không sửa dữ liệu thật; bài ghi dùng database tạm.